## highz-accretion-atlas v1 evaluation

Build v1 science tables from `data/processed/v1_processed.csv` using the README growth assumptions.

In [ ]:
from pathlib import Path
import re
import sys

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository root from the current notebook or script directory."""
    current = Path.cwd() if start is None else Path(start)
    for candidate in (current.resolve(), *current.resolve().parents):
        if (candidate / 'README.md').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('Could not locate repository root')


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / 'src'
RESULTS_DIR = REPO_ROOT / 'results'
PROCESSED_DATA_PATH = REPO_ROOT / 'data' / 'processed' / 'v1_processed.csv'

PARAMETER_MAP_DIR = RESULTS_DIR / 'v1_parameter_maps'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PARAMETER_MAP_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC_DIR))

EVALUATION_TABLE_PATH = RESULTS_DIR / 'v1_evaluation_table.csv'
REQUIRED_FEDD_TABLE_PATH = RESULTS_DIR / 'v1_required_fedd_by_seed_mass.csv'
REQUIRED_MSEED_TABLE_PATH = RESULTS_DIR / 'v1_required_mseed_by_growth_assumption.csv'
SAMPLE_SUMMARY_PATH = RESULTS_DIR / 'v1_sample_summary.csv'
GROWTH_TRACK_PNG_PATH = RESULTS_DIR / 'v1_mbh_vs_redshift_growth_tracks.png'
SAMPLE_SUMMARY_PNG_PATH = RESULTS_DIR / 'v1_sample_compatibility_summary.png'

print(f'Repository root: {REPO_ROOT}')
print(f'Processed catalogue: {PROCESSED_DATA_PATH}')

In [ ]:
from models import (
    SEED_MODELS,
    apply_mbh_interpretation,
    apply_mstar_agn_contamination,
    available_growth_time_gyr,
    evaluate_seed_model,
    growth_parameter_grid,
    predicted_log_mbh,
    required_fedd_for_seed,
    required_seed_mass_for_growth,
    run_growth_sanity_checks,
)
from scoring import score_model_table


df = pd.read_csv(PROCESSED_DATA_PATH)
if df.empty:
    raise ValueError('Processed v1 catalogue is empty')
if not df['measurement_id'].is_unique:
    raise ValueError('Processed v1 catalogue has duplicate measurement_id values')

growth_checks = run_growth_sanity_checks()
print(f'Loaded {len(df)} processed rows')
print('Growth sanity checks passed:', growth_checks)

In [ ]:
# Keep the v1 interpretation variants from the original notebook.
INTERPRETATION_VARIANTS = {
    'baseline': {'mbh_delta_dex': 0.0, 'mstar_agn_fraction': 0.0},
    'mbh_minus_0p3dex': {'mbh_delta_dex': -0.3, 'mstar_agn_fraction': 0.0},
    'mbh_plus_0p3dex': {'mbh_delta_dex': 0.3, 'mstar_agn_fraction': 0.0},
    'mstar_agn_20pct': {'mbh_delta_dex': 0.0, 'mstar_agn_fraction': 0.2},
}

# Current v1 growth assumptions used for required seed-mass tables.
GROWTH_CONFIGS = {
    'eddington_eps0p1': {'f_edd_avg': 1.0, 'epsilon': 0.1, 'merger_boost': 1.0},
    'subeddington_eps0p1': {'f_edd_avg': 0.3, 'epsilon': 0.1, 'merger_boost': 1.0},
    'supercritical_eps0p05': {'f_edd_avg': 2.0, 'epsilon': 0.05, 'merger_boost': 1.0},
    'merger_boost_x2': {'f_edd_avg': 1.0, 'epsilon': 0.1, 'merger_boost': 2.0},
}

# README interpretability thresholds for required f_Edd tables.
SEED_MASS_ASSUMPTIONS = {
    'seed_1e2_msun': 2.0,
    'seed_1e4_msun': 4.0,
    'seed_1e5_msun': 5.0,
}

# Unique epsilon/merger cases for solving required f_Edd. f_Edd itself is the unknown here.
FEDD_REQUIREMENT_CONFIGS = {
    'eps0p1_no_merger_boost': {'epsilon': 0.1, 'merger_boost': 1.0},
    'eps0p05_no_merger_boost': {'epsilon': 0.05, 'merger_boost': 1.0},
    'eps0p1_merger_boost_x2': {'epsilon': 0.1, 'merger_boost': 2.0},
}

Z_SEED_V1 = 30.0
OBJECT_METADATA_COLUMNS = [
    'measurement_id', 'object_id', 'redshift', 'redshift_kind', 'survey',
    'object_class', 'quality_flag', 'source_key', 'source_table',
    'missing_mstar_flag', 'missing_lbol_flag', 'missing_edd_ratio_flag',
    'missing_lensing_flag',
]

CB_PALETTE = {
    'blue': '#0072B2',
    'orange': '#E69F00',
    'green': '#009E73',
    'red': '#D55E00',
    'purple': '#CC79A7',
    'sky': '#56B4E9',
    'yellow': '#F0E442',
    'black': '#222222',
}

PRETTY_GROWTH = {
    'eddington_eps0p1': r'$f_{\rm Edd}=1$, $\epsilon=0.1$',
    'subeddington_eps0p1': r'$f_{\rm Edd}=0.3$, $\epsilon=0.1$',
    'supercritical_eps0p05': r'$f_{\rm Edd}=2$, $\epsilon=0.05$',
    'merger_boost_x2': r'$f_{\rm Edd}=1$, $\epsilon=0.1$, $B_{\rm merge}=2$',
}
PRETTY_SEED_MODEL = {
    'light_popiii': 'Light seeds',
    'intermediate_cluster': 'Cluster seeds',
    'heavy_dcbh': 'Heavy seeds',
    'pbh': 'PBH seeds',
}
PRETTY_SEED_MASS = {
    'seed_1e2_msun': r'$10^2\,M_\odot$',
    'seed_1e4_msun': r'$10^4\,M_\odot$',
    'seed_1e5_msun': r'$10^5\,M_\odot$',
}

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 350,
    'savefig.bbox': 'tight',
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'legend.fontsize': 8.5,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.linewidth': 0.9,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    'grid.color': '#D8D8D8',
    'grid.linewidth': 0.6,
    'grid.alpha': 0.55,
})

In [ ]:
def object_metadata(obj: pd.Series) -> dict[str, object]:
    return {col: obj[col] for col in OBJECT_METADATA_COLUMNS if col in obj.index}


def evaluated_masses(obj: pd.Series, variant: dict[str, float]) -> tuple[float, float, float]:
    log_mbh = float(apply_mbh_interpretation(obj['log_mbh_msun_std'], variant['mbh_delta_dex']))
    log_mstar = float(apply_mstar_agn_contamination(obj['log_mstar_msun_std'], variant['mstar_agn_fraction']))
    return log_mbh, log_mstar, log_mbh - log_mstar


def fedd_label(required_fedd: float) -> str:
    if pd.isna(required_fedd):
        return 'missing'
    if required_fedd <= 1.0:
        return 'eddington_or_below'
    if required_fedd <= 3.0:
        return 'super_eddington'
    return 'extreme'


def seed_label(required_log_mseed: float) -> str:
    if pd.isna(required_log_mseed):
        return 'missing'
    if required_log_mseed <= 2.0:
        return 'light_seed_scale'
    if required_log_mseed <= 4.0:
        return 'intermediate_seed_scale'
    if required_log_mseed <= 6.0:
        return 'heavy_seed_scale'
    return 'above_heavy_seed_scale'


def score_label(score: float) -> str:
    if pd.isna(score):
        return 'missing'
    if score >= 0.8:
        return 'strong'
    if score >= 0.5:
        return 'partial'
    return 'poor'


def safe_filename(value: object) -> str:
    text = re.sub(r'[^A-Za-z0-9]+', '-', str(value)).strip('-')
    return text.lower() or 'object'


def save_figure(fig: plt.Figure, png_path: Path) -> None:
    fig.savefig(png_path, dpi=350, facecolor='white')
    plt.close(fig)

In [ ]:
rows = []

for _, obj in df.iterrows():
    delta_t = float(available_growth_time_gyr(z_seed=Z_SEED_V1, z_obs=obj['redshift']))
    base_metadata = object_metadata(obj)

    for variant_name, variant in INTERPRETATION_VARIANTS.items():
        log_mbh, log_mstar, log_ratio = evaluated_masses(obj, variant)

        for growth_name, growth in GROWTH_CONFIGS.items():
            for seed_model_name, seed_model in SEED_MODELS.items():
                log_mseed_mid = 0.5 * (seed_model.log_mseed_min + seed_model.log_mseed_max)
                required_fedd = float(
                    required_fedd_for_seed(
                        log_mseed=log_mseed_mid,
                        log_mbh_final=log_mbh,
                        epsilon=growth['epsilon'],
                        z_seed=Z_SEED_V1,
                        z_obs=obj['redshift'],
                        merger_boost=growth['merger_boost'],
                    )
                )
                seed_eval = evaluate_seed_model(
                    log_mbh_final=log_mbh,
                    delta_t_gyr=delta_t,
                    model_name=seed_model_name,
                    f_edd_avg=growth['f_edd_avg'],
                    epsilon=growth['epsilon'],
                    merger_boost=growth['merger_boost'],
                )

                rows.append({
                    **base_metadata,
                    'z_seed': Z_SEED_V1,
                    'delta_t_gyr': delta_t,
                    'interpretation_variant': variant_name,
                    'mbh_delta_dex': variant['mbh_delta_dex'],
                    'mstar_agn_fraction': variant['mstar_agn_fraction'],
                    'log_mbh_eval': log_mbh,
                    'log_mstar_eval': log_mstar,
                    'log_mbh_mstar_ratio_eval': log_ratio,
                    'growth_config': growth_name,
                    'f_edd_avg': growth['f_edd_avg'],
                    'epsilon': growth['epsilon'],
                    'merger_boost': growth['merger_boost'],
                    'seed_model': seed_model_name,
                    'seed_assumption_log_mseed_mid': log_mseed_mid,
                    'seed_assumption_mseed_mid_msun': 10 ** log_mseed_mid,
                    'required_fedd': required_fedd,
                    **seed_eval,
                })

evaluation_df = score_model_table(pd.DataFrame(rows))
evaluation_df['required_fedd_label'] = evaluation_df['required_fedd'].map(fedd_label)
evaluation_df['required_mseed_label'] = evaluation_df['required_log_mseed'].map(seed_label)
evaluation_df['feasibility_label'] = evaluation_df['feasibility_score'].map(score_label)
evaluation_df.to_csv(EVALUATION_TABLE_PATH, index=False)

print(f'Saved {len(evaluation_df)} rows: {EVALUATION_TABLE_PATH}')
evaluation_df.head()

In [ ]:
rows = []

for _, obj in df.iterrows():
    delta_t = float(available_growth_time_gyr(z_seed=Z_SEED_V1, z_obs=obj['redshift']))
    base_metadata = object_metadata(obj)

    for variant_name, variant in INTERPRETATION_VARIANTS.items():
        log_mbh, log_mstar, log_ratio = evaluated_masses(obj, variant)

        for config_name, config in FEDD_REQUIREMENT_CONFIGS.items():
            for seed_label_name, log_mseed in SEED_MASS_ASSUMPTIONS.items():
                required_fedd = float(
                    required_fedd_for_seed(
                        log_mseed=log_mseed,
                        log_mbh_final=log_mbh,
                        epsilon=config['epsilon'],
                        z_seed=Z_SEED_V1,
                        z_obs=obj['redshift'],
                        merger_boost=config['merger_boost'],
                    )
                )

                rows.append({
                    **base_metadata,
                    'z_seed': Z_SEED_V1,
                    'delta_t_gyr': delta_t,
                    'interpretation_variant': variant_name,
                    'mbh_delta_dex': variant['mbh_delta_dex'],
                    'mstar_agn_fraction': variant['mstar_agn_fraction'],
                    'log_mbh_eval': log_mbh,
                    'log_mstar_eval': log_mstar,
                    'log_mbh_mstar_ratio_eval': log_ratio,
                    'fedd_requirement_config': config_name,
                    'epsilon': config['epsilon'],
                    'merger_boost': config['merger_boost'],
                    'seed_mass_assumption': seed_label_name,
                    'log_mseed_assumption': log_mseed,
                    'mseed_assumption_msun': 10 ** log_mseed,
                    'required_fedd': required_fedd,
                    'required_fedd_label': fedd_label(required_fedd),
                })

required_fedd_df = pd.DataFrame(rows)
required_fedd_df.to_csv(REQUIRED_FEDD_TABLE_PATH, index=False)

print(f'Saved {len(required_fedd_df)} rows: {REQUIRED_FEDD_TABLE_PATH}')
required_fedd_df.head()

In [ ]:
rows = []

for _, obj in df.iterrows():
    delta_t = float(available_growth_time_gyr(z_seed=Z_SEED_V1, z_obs=obj['redshift']))
    base_metadata = object_metadata(obj)

    for variant_name, variant in INTERPRETATION_VARIANTS.items():
        log_mbh, log_mstar, log_ratio = evaluated_masses(obj, variant)

        for growth_name, growth in GROWTH_CONFIGS.items():
            required_log_mseed = float(
                required_seed_mass_for_growth(
                    log_mbh_final=log_mbh,
                    f_edd=growth['f_edd_avg'],
                    epsilon=growth['epsilon'],
                    z_seed=Z_SEED_V1,
                    z_obs=obj['redshift'],
                    merger_boost=growth['merger_boost'],
                )
            )

            rows.append({
                **base_metadata,
                'z_seed': Z_SEED_V1,
                'delta_t_gyr': delta_t,
                'interpretation_variant': variant_name,
                'mbh_delta_dex': variant['mbh_delta_dex'],
                'mstar_agn_fraction': variant['mstar_agn_fraction'],
                'log_mbh_eval': log_mbh,
                'log_mstar_eval': log_mstar,
                'log_mbh_mstar_ratio_eval': log_ratio,
                'growth_config': growth_name,
                'f_edd_avg': growth['f_edd_avg'],
                'epsilon': growth['epsilon'],
                'merger_boost': growth['merger_boost'],
                'required_log_mseed': required_log_mseed,
                'required_mseed_msun': 10 ** required_log_mseed,
                'required_mseed_label': seed_label(required_log_mseed),
            })

required_mseed_df = pd.DataFrame(rows)
required_mseed_df.to_csv(REQUIRED_MSEED_TABLE_PATH, index=False)

print(f'Saved {len(required_mseed_df)} rows: {REQUIRED_MSEED_TABLE_PATH}')
required_mseed_df.head()

In [ ]:
sample_summary_df = (
    evaluation_df
    .groupby(['interpretation_variant', 'growth_config', 'seed_model'], as_index=False)
    .agg(
        n_rows=('measurement_id', 'size'),
        n_objects=('measurement_id', 'nunique'),
        median_required_fedd=('required_fedd', 'median'),
        max_required_fedd=('required_fedd', 'max'),
        median_required_log_mseed=('required_log_mseed', 'median'),
        min_required_log_mseed=('required_log_mseed', 'min'),
        max_required_log_mseed=('required_log_mseed', 'max'),
        feasible_seed_model_fraction=('is_feasible', 'mean'),
        mean_feasibility_score=('feasibility_score', 'mean'),
        median_feasibility_score=('feasibility_score', 'median'),
    )
    .sort_values(['interpretation_variant', 'growth_config', 'seed_model'])
)
sample_summary_df.to_csv(SAMPLE_SUMMARY_PATH, index=False)

print(f'Saved {len(sample_summary_df)} rows: {SAMPLE_SUMMARY_PATH}')
sample_summary_df.head()

In [ ]:
# Paper-ready M_BH versus redshift figure with simple growth tracks.
fig, ax = plt.subplots(figsize=(7.7, 5.35), constrained_layout=True)

track_z = np.linspace(10.0, 4.0, 300)
track_specs = [
    (2.0, r'$10^2\,M_\odot$', CB_PALETTE['blue']),
    (4.0, r'$10^4\,M_\odot$', CB_PALETTE['green']),
    (5.0, r'$10^5\,M_\odot$', CB_PALETTE['orange']),
]
fedd_track_specs = [
    (0.3, (0, (5, 3)), r'$f_{\rm Edd}=0.3$'),
    (1.0, '-', r'$f_{\rm Edd}=1$'),
    (2.0, (0, (1.2, 2.0)), r'$f_{\rm Edd}=2$'),
]
for log_seed, seed_label, color in track_specs:
    for f_edd_track, linestyle, _ in fedd_track_specs:
        track_mass = predicted_log_mbh(
            log_mseed=log_seed,
            f_edd=f_edd_track,
            epsilon=0.1,
            z_seed=Z_SEED_V1,
            z_obs=track_z,
        )
        ax.plot(track_z, track_mass, color=color, lw=2.0, ls=linestyle, alpha=0.95)

quality_styles = {
    'robust': {'marker': 'o', 'facecolor': CB_PALETTE['black'], 'edgecolor': 'white', 'label': 'Robust v1 objects'},
    'tentative': {'marker': 's', 'facecolor': 'white', 'edgecolor': CB_PALETTE['red'], 'label': 'Tentative v1 objects'},
}
for quality, subset in df.groupby('quality_flag'):
    style = quality_styles.get(quality, quality_styles['robust'])
    yerr = np.vstack([
        subset['log_mbh_err_minus_std'].fillna(0.0).to_numpy(float),
        subset['log_mbh_err_plus_std'].fillna(0.0).to_numpy(float),
    ])
    ax.errorbar(
        subset['redshift'],
        subset['log_mbh_msun_std'],
        yerr=yerr,
        fmt=style['marker'],
        ms=5.6,
        mfc=style['facecolor'],
        mec=style['edgecolor'],
        mew=0.9,
        ecolor='#6F6F6F',
        elinewidth=0.8,
        capsize=1.8,
        alpha=0.94,
        label=style['label'],
        zorder=4,
    )

ax.axhspan(4.0, 6.0, color='#F2F2F2', zorder=0, label='Heavy-seed scale')
ax.set_xlim(10.1, 3.9)
ax.set_ylim(4.7, 10.0)
ax.set_xlabel('Redshift, $z$')
ax.set_ylabel(r'$\log_{10}(M_{\rm BH}/M_\odot)$')
ax.set_title(r'v1 black-hole masses and constant-accretion growth tracks')
ax.grid(True, which='major')
seed_handles = [Line2D([0], [0], color=color, lw=2.2, label=label) for _, label, color in track_specs]
fedd_handles = [Line2D([0], [0], color='#4A4A4A', lw=2.0, ls=linestyle, label=label) for _, linestyle, label in fedd_track_specs]
object_handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor=CB_PALETTE['black'], markeredgecolor='white', markersize=6, label='Robust v1 objects'),
    Line2D([0], [0], marker='s', color='none', markerfacecolor='white', markeredgecolor=CB_PALETTE['red'], markersize=6, label='Tentative v1 objects'),
    Line2D([0], [0], color='#F2F2F2', lw=8, label='Heavy-seed scale'),
]
legend1 = ax.legend(handles=seed_handles, title='Seed mass', loc='upper right', frameon=False)
ax.add_artist(legend1)
legend2 = ax.legend(handles=fedd_handles + object_handles, loc='lower left', frameon=False, ncols=1)
ax.add_artist(legend2)
fig.text(
    0.02,
    -0.035,
    'Caption: points are the v1 catalogue black-hole masses with reported mass uncertainties. Curves start at z_seed = 30; color marks seed mass and line style marks the assumed constant average accretion rate. Objects above a curve require a heavier seed, higher average accretion, lower radiative efficiency, or extra growth channels under this v1 model.',
    ha='left',
    va='top',
    fontsize=8.2,
    color='#333333',
    wrap=True,
)
save_figure(fig, GROWTH_TRACK_PNG_PATH)
print(f'Saved growth-track figure: {GROWTH_TRACK_PNG_PATH.relative_to(REPO_ROOT)}')

In [ ]:
# Per-object f_Edd versus seed-mass parameter maps.
for old_map in PARAMETER_MAP_DIR.glob('v1_parameter_map_*.png'):
    old_map.unlink()
log_mseed_axis = np.linspace(1.0, 6.2, 180)
f_edd_axis = np.linspace(0.0, 3.0, 180)
map_paths = []

for _, obj in df.sort_values(['redshift', 'object_id']).iterrows():
    grid = growth_parameter_grid(
        log_mseed_values=log_mseed_axis,
        f_edd_values=f_edd_axis,
        epsilon=0.1,
        z_seed=Z_SEED_V1,
        z_obs=obj['redshift'],
    )
    x_grid = grid['log_mseed_grid']
    y_grid = grid['f_edd_grid']
    z_grid = grid['predicted_log_mbh']
    observed = float(obj['log_mbh_msun_std'])

    fig, ax = plt.subplots(figsize=(6.4, 5.65), constrained_layout=True)
    mesh = ax.pcolormesh(
        x_grid,
        y_grid,
        z_grid,
        shading='auto',
        cmap='viridis',
        vmin=5.0,
        vmax=10.0,
        rasterized=True,
    )
    cbar = fig.colorbar(mesh, ax=ax, pad=0.018)
    cbar.set_label(r'Predicted $\log_{10}(M_{\rm BH}/M_\odot)$')

    z_min = float(np.nanmin(z_grid))
    z_max = float(np.nanmax(z_grid))
    if z_min <= observed <= z_max:
        contour = ax.contour(x_grid, y_grid, z_grid, levels=[observed], colors='white', linewidths=2.0)
        ax.clabel(contour, fmt={observed: 'Observed'}, inline=True, fontsize=8.5)

    err_minus = obj.get('log_mbh_err_minus_std', np.nan)
    err_plus = obj.get('log_mbh_err_plus_std', np.nan)
    if pd.notna(err_minus) and pd.notna(err_plus):
        uncertainty_levels = [observed - float(err_minus), observed + float(err_plus)]
        uncertainty_levels = [level for level in uncertainty_levels if z_min <= level <= z_max]
        if uncertainty_levels:
            ax.contour(
                x_grid,
                y_grid,
                z_grid,
                levels=sorted(uncertainty_levels),
                colors='white',
                linestyles='--',
                linewidths=1.0,
                alpha=0.72,
            )

    ax.axhline(1.0, color='white', lw=1.3, ls='-', alpha=0.9)
    ax.text(1.06, 1.05, r'$f_{\rm Edd}=1$', color='white', fontsize=8.5, va='bottom')
    for x_ref, label in [(2.0, r'$10^2$'), (4.0, r'$10^4$'), (5.0, r'$10^5$')]:
        ax.axvline(x_ref, color='white', lw=0.9, ls=':', alpha=0.82)
        ax.text(x_ref + 0.03, 2.92, label, color='white', fontsize=8.0, rotation=90, va='top')

    quality_text = str(obj.get('quality_flag', '')).title()
    title = f"{obj['object_id']}  |  z = {float(obj['redshift']):.3f}  |  {quality_text}"
    ax.set_title(title)
    ax.set_xlabel(r'$\log_{10}(M_{\rm seed}/M_\odot)$')
    ax.set_ylabel(r'Average $f_{\rm Edd}$')
    ax.set_xlim(log_mseed_axis.min(), log_mseed_axis.max())
    ax.set_ylim(f_edd_axis.min(), f_edd_axis.max())
    ax.grid(False)
    fig.text(
        0.02,
        -0.035,
        'Caption: color gives the black-hole mass predicted by each seed/accretion pair. The solid white contour reproduces the observed mass; dashed contours show the reported mass uncertainty. The dotted vertical lines mark the README seed thresholds (10^2, 10^4, 10^5 solar masses), so the contour can be read against light, intermediate, and heavy seed scales.',
        ha='left',
        va='top',
        fontsize=7.8,
        color='#333333',
        wrap=True,
    )

    stem = f"v1_parameter_map_{safe_filename(obj['measurement_id'])}"
    png_path = PARAMETER_MAP_DIR / f'{stem}.png'
    save_figure(fig, png_path)
    map_paths.append(png_path)

print(f'Saved {len(map_paths)} per-object parameter maps in {PARAMETER_MAP_DIR.relative_to(REPO_ROOT)}')

In [ ]:
# Sample-level compatibility and requirement summary.
baseline_summary = sample_summary_df[sample_summary_df['interpretation_variant'] == 'baseline'].copy()
heatmap = baseline_summary.pivot(index='growth_config', columns='seed_model', values='feasible_seed_model_fraction')
heatmap = heatmap.loc[list(GROWTH_CONFIGS), list(SEED_MODELS)]

baseline_fedd = required_fedd_df[
    (required_fedd_df['interpretation_variant'] == 'baseline')
    & (required_fedd_df['fedd_requirement_config'] == 'eps0p1_no_merger_boost')
]
fedd_summary = (
    baseline_fedd
    .groupby('seed_mass_assumption')['required_fedd']
    .agg(median='median', p16=lambda s: np.percentile(s, 16), p84=lambda s: np.percentile(s, 84))
    .loc[list(SEED_MASS_ASSUMPTIONS)]
)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10.8, 5.05), constrained_layout=True)

im = ax0.imshow(heatmap.to_numpy(float), vmin=0.0, vmax=1.0, cmap='viridis', aspect='auto')
ax0.set_xticks(np.arange(len(heatmap.columns)))
ax0.set_xticklabels([PRETTY_SEED_MODEL[col] for col in heatmap.columns], rotation=25, ha='right')
ax0.set_yticks(np.arange(len(heatmap.index)))
ax0.set_yticklabels([PRETTY_GROWTH[idx] for idx in heatmap.index])
ax0.set_title('Compatible-object fraction')
for y in range(heatmap.shape[0]):
    for x in range(heatmap.shape[1]):
        value = heatmap.iloc[y, x]
        color = 'white' if value < 0.55 else '#1A1A1A'
        ax0.text(x, y, f'{value:.2f}', ha='center', va='center', color=color, fontsize=8.5)
cbar = fig.colorbar(im, ax=ax0, pad=0.018)
cbar.set_label('Fraction of v1 objects\ncompatible with seed model')

x = np.arange(len(fedd_summary))
median = fedd_summary['median'].to_numpy(float)
lower = median - fedd_summary['p16'].to_numpy(float)
upper = fedd_summary['p84'].to_numpy(float) - median
ax1.errorbar(
    x,
    median,
    yerr=np.vstack([lower, upper]),
    fmt='o',
    ms=6,
    lw=1.2,
    capsize=3,
    color=CB_PALETTE['blue'],
    ecolor=CB_PALETTE['blue'],
)
ax1.axhline(1.0, color=CB_PALETTE['red'], ls='--', lw=1.2, label=r'$f_{\rm Edd}=1$')
ax1.set_yscale('log')
ax1.set_xticks(x)
ax1.set_xticklabels([PRETTY_SEED_MASS[idx] for idx in fedd_summary.index])
ax1.set_ylabel(r'Required average $f_{\rm Edd}$')
ax1.set_xlabel(r'Assumed seed mass')
ax1.set_title(r'Baseline requirements, $\epsilon=0.1$')
ax1.grid(True, which='major')
ax1.legend(frameon=False, loc='best')
fig.text(
    0.02,
    -0.035,
    'Caption: left panel shows the fraction of v1 objects whose required seed mass falls inside each seed-model range for the stated growth assumption; this is a compatibility diagnostic, not a probability. Right panel shows the median and 16th-84th percentile required average f_Edd for fixed seed masses in the baseline epsilon = 0.1 case.',
    ha='left',
    va='top',
    fontsize=8.2,
    color='#333333',
    wrap=True,
)

save_figure(fig, SAMPLE_SUMMARY_PNG_PATH)
print(f'Saved sample summary figure: {SAMPLE_SUMMARY_PNG_PATH.relative_to(REPO_ROOT)}')

In [ ]:
expected_counts = {
    'evaluation': len(df) * len(INTERPRETATION_VARIANTS) * len(GROWTH_CONFIGS) * len(SEED_MODELS),
    'required_fedd': len(df) * len(INTERPRETATION_VARIANTS) * len(FEDD_REQUIREMENT_CONFIGS) * len(SEED_MASS_ASSUMPTIONS),
    'required_mseed': len(df) * len(INTERPRETATION_VARIANTS) * len(GROWTH_CONFIGS),
    'sample_summary': len(INTERPRETATION_VARIANTS) * len(GROWTH_CONFIGS) * len(SEED_MODELS),
}

actual_counts = {
    'evaluation': len(evaluation_df),
    'required_fedd': len(required_fedd_df),
    'required_mseed': len(required_mseed_df),
    'sample_summary': len(sample_summary_df),
}
if expected_counts != actual_counts:
    raise AssertionError(f'Unexpected row counts: expected {expected_counts}, got {actual_counts}')

duplicate_checks = {
    'evaluation': evaluation_df.duplicated(['measurement_id', 'interpretation_variant', 'growth_config', 'seed_model']).sum(),
    'required_fedd': required_fedd_df.duplicated(['measurement_id', 'interpretation_variant', 'fedd_requirement_config', 'seed_mass_assumption']).sum(),
    'required_mseed': required_mseed_df.duplicated(['measurement_id', 'interpretation_variant', 'growth_config']).sum(),
    'sample_summary': sample_summary_df.duplicated(['interpretation_variant', 'growth_config', 'seed_model']).sum(),
}
if any(count != 0 for count in duplicate_checks.values()):
    raise AssertionError(f'Accidental duplicate rows found: {duplicate_checks}')

figure_paths = [GROWTH_TRACK_PNG_PATH, SAMPLE_SUMMARY_PNG_PATH]
parameter_map_pngs = sorted(PARAMETER_MAP_DIR.glob('v1_parameter_map_*.png'))
if len(parameter_map_pngs) != len(df):
    raise AssertionError(
        f'Expected {len(df)} PNG parameter maps, got {len(parameter_map_pngs)}'
    )

for path in [EVALUATION_TABLE_PATH, REQUIRED_FEDD_TABLE_PATH, REQUIRED_MSEED_TABLE_PATH, SAMPLE_SUMMARY_PATH, *figure_paths]:
    if not path.exists():
        raise FileNotFoundError(path)

print('Expected row counts:', expected_counts)
print('Duplicate checks:', duplicate_checks)
print('Generated CSVs:')
for path in [EVALUATION_TABLE_PATH, REQUIRED_FEDD_TABLE_PATH, REQUIRED_MSEED_TABLE_PATH, SAMPLE_SUMMARY_PATH]:
    print(f'  {path.relative_to(REPO_ROOT)}')
print('Generated figures:')
for path in figure_paths:
    print(f'  {path.relative_to(REPO_ROOT)}')
print(f'  {len(parameter_map_pngs)} PNG parameter maps in {PARAMETER_MAP_DIR.relative_to(REPO_ROOT)}')